# Face Recognition - FaceAttend
**Model**: ArcFace (Pretrained via DeepFace, trained on MS-Celeb-1M)

**Metode**: Transfer Learning / Feature Extraction + Cosine Similarity

**Alur**:
1. Install dependencies
2. Mount Google Drive (dataset ada di sini)
3. Training — ekstrak embedding, simpan ke JSON
4. Testing — evaluasi akurasi, precision, recall, F1, confusion matrix

## 1. Install Dependencies

In [ ]:
!pip install deepface tf-keras numpy matplotlib seaborn

## 2. Mount Google Drive

Pastikan folder dataset sudah diupload ke Google Drive dengan struktur:
```
MyDrive/
└── ml_model/
    └── dataset/
        ├── ghani/
        │   ├── foto1.jpg
        │   └── ...
        └── Radit/
            └── ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Konfigurasi Path & Parameter

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from deepface import DeepFace

# ── Ubah path ini sesuai lokasi di Google Drive kamu ──
DATASET_DIR    = "/content/drive/MyDrive/ml_model/dataset"
EMBEDDINGS_FILE = "/content/drive/MyDrive/ml_model/embeddings.json"
TEST_SPLIT_FILE = "/content/drive/MyDrive/ml_model/test_split.json"
RESULTS_FILE    = "/content/drive/MyDrive/ml_model/test_results.json"

# ── Model & Threshold ──────────────────────────────────
MODEL_NAME   = "ArcFace"   # Pretrained model
TRAIN_RATIO  = 0.8         # 80% train, 20% test
THRESHOLD    = 0.68        # Minimum cosine similarity
MATCH_MARGIN = 0.05        # Selisih minimum skor #1 vs #2
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Dataset dir : {DATASET_DIR}")
print(f"Model       : {MODEL_NAME}")
print(f"Threshold   : {THRESHOLD}")
print(f"Margin      : {MATCH_MARGIN}")
print(f"Train ratio : {TRAIN_RATIO}")

## 4. Helper Functions

In [ ]:
def get_embedding(img_path):
    """Extract face embedding dari satu gambar menggunakan ArcFace."""
    try:
        result = DeepFace.represent(
            img_path=img_path,
            model_name=MODEL_NAME,
            enforce_detection=False,
        )
        if result:
            return result[0]["embedding"]
    except Exception as e:
        print(f"  ⚠ Gagal: {img_path} → {e}")
    return None


def cosine_similarity(a, b):
    """Hitung cosine similarity antara dua vektor."""
    a = np.array(a, dtype=np.float64)
    b = np.array(b, dtype=np.float64)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def predict(face_embedding, embeddings):
    """
    Prediksi identitas dari embedding.
    Returns: (predicted_label, best_score, second_score)
    """
    best_label   = None
    best_score   = -1.0
    second_score = -1.0

    for person_name, data in embeddings.items():
        score = cosine_similarity(face_embedding, data["embedding"])
        if score > best_score:
            second_score = best_score
            best_score   = score
            best_label   = person_name
        elif score > second_score:
            second_score = score

    if second_score < 0:
        second_score = 0.0

    if (
        best_label
        and best_score >= THRESHOLD
        and (best_score - second_score) >= MATCH_MARGIN
    ):
        return best_label, best_score, second_score

    return None, best_score, second_score


def compute_metrics(y_true, y_pred, labels):
    """Hitung Accuracy, Precision, Recall, F1 (macro average)."""
    n = len(labels)
    label_to_idx = {l: i for i, l in enumerate(labels)}
    cm = np.zeros((n, n + 1), dtype=int)

    for true, pred in zip(y_true, y_pred):
        true_idx = label_to_idx.get(true, -1)
        if true_idx == -1:
            continue
        pred_idx = label_to_idx.get(pred, n) if pred else n
        cm[true_idx][pred_idx] += 1

    precisions, recalls, f1s = [], [], []
    for i in range(n):
        tp = cm[i][i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    correct  = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    accuracy = correct / len(y_true) if y_true else 0.0

    return {
        "accuracy"        : accuracy,
        "macro_precision" : float(np.mean(precisions)),
        "macro_recall"    : float(np.mean(recalls)),
        "macro_f1"        : float(np.mean(f1s)),
        "per_class"       : {
            labels[i]: {
                "precision": precisions[i],
                "recall"   : recalls[i],
                "f1"       : f1s[i],
            } for i in range(n)
        },
        "confusion_matrix": cm,
        "cm_labels"       : labels + ["unknown"],
    }

print("✅ Helper functions loaded.")

## 5. Training — Feature Extraction & Embedding Generation

In [ ]:
print("=" * 60)
print("  TRAINING — FEATURE EXTRACTION")
print(f"  Model  : {MODEL_NAME} (Pretrained - DeepFace)")
print(f"  Method : Transfer Learning / Feature Extraction")
print("=" * 60)

embeddings    = {}
test_split    = {}
train_summary = []

persons = [
    p for p in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, p))
]
print(f"\n📂 Ditemukan {len(persons)} orang: {persons}\n")

for person_name in persons:
    person_dir = os.path.join(DATASET_DIR, person_name)
    photos = [
        f for f in os.listdir(person_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    if not photos:
        print(f"  ⚠ {person_name}: tidak ada foto, skip.")
        continue

    random.shuffle(photos)
    split_idx    = max(1, int(len(photos) * TRAIN_RATIO))
    train_photos = photos[:split_idx]
    test_photos  = photos[split_idx:]

    print(f"👤 {person_name} — train: {len(train_photos)}, test: {len(test_photos)}")

    train_embeddings = []
    for photo in train_photos:
        path = os.path.join(person_dir, photo)
        emb  = get_embedding(path)
        if emb is not None:
            train_embeddings.append(emb)
            print(f"   ✅ {photo}")
        else:
            print(f"   ❌ {photo}")

    if not train_embeddings:
        print(f"   ❌ Skip {person_name}\n")
        continue

    avg_embedding = np.mean(train_embeddings, axis=0).tolist()
    embeddings[person_name] = {
        "embedding"  : avg_embedding,
        "photo_count": len(train_embeddings),
        "model"      : MODEL_NAME,
    }
    test_split[person_name] = [
        os.path.join(person_dir, p) for p in test_photos
    ]
    train_summary.append({
        "person"     : person_name,
        "train"      : len(train_embeddings),
        "test"       : len(test_photos),
        "emb_dim"    : len(avg_embedding),
    })
    print(f"   ✅ Embedding selesai (dim={len(avg_embedding)})\n")

# Simpan
with open(EMBEDDINGS_FILE, "w") as f:
    json.dump(embeddings, f, indent=2)
with open(TEST_SPLIT_FILE, "w") as f:
    json.dump(test_split, f, indent=2)

print("=" * 60)
print("  TRAINING SELESAI")
print(f"  {'Nama':<20} {'Train':>6} {'Test':>6} {'Dim':>6}")
print(f"  {'-'*40}")
for s in train_summary:
    print(f"  {s['person']:<20} {s['train']:>6} {s['test']:>6} {s['emb_dim']:>6}")
print("=" * 60)

## 6. Testing — Evaluasi Model

In [ ]:
print("=" * 60)
print("  TESTING — EVALUASI MODEL")
print(f"  Threshold : {THRESHOLD}")
print(f"  Margin    : {MATCH_MARGIN}")
print("=" * 60)

labels     = list(embeddings.keys())
y_true     = []
y_pred     = []
detail_rows = []

for true_label, photo_paths in test_split.items():
    if not photo_paths:
        continue
    print(f"\n🔍 Testing: {true_label} ({len(photo_paths)} foto)")
    for path in photo_paths:
        emb = get_embedding(path)
        if emb is None:
            continue
        pred_label, best_sc, second_sc = predict(emb, embeddings)
        icon = "✅" if pred_label == true_label else "❌"
        print(f"   {icon} {os.path.basename(path):<30} "
              f"pred={pred_label or 'unknown':<15} "
              f"score={best_sc:.4f}  margin={best_sc - second_sc:.4f}")
        y_true.append(true_label)
        y_pred.append(pred_label)
        detail_rows.append({
            "file"      : os.path.basename(path),
            "true"      : true_label,
            "predicted" : pred_label or "unknown",
            "correct"   : pred_label == true_label,
            "best_score": round(best_sc, 4),
            "margin"    : round(best_sc - second_sc, 4),
        })

metrics = compute_metrics(y_true, y_pred, labels)

print("\n" + "=" * 60)
print("  HASIL EVALUASI")
print("=" * 60)
print(f"  Total test    : {len(y_true)}")
print(f"  Benar         : {sum(1 for t,p in zip(y_true,y_pred) if t==p)}")
print(f"  Tidak dikenali: {sum(1 for p in y_pred if p is None)}")
print()
print(f"  Accuracy      : {metrics['accuracy']*100:.2f}%")
print(f"  Precision     : {metrics['macro_precision']*100:.2f}%")
print(f"  Recall        : {metrics['macro_recall']*100:.2f}%")
print(f"  F1-Score      : {metrics['macro_f1']*100:.2f}%")
print()
print(f"  {'Kelas':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*52}")
for label, m in metrics["per_class"].items():
    print(f"  {label:<20} {m['precision']*100:>9.2f}% "
          f"{m['recall']*100:>9.2f}% {m['f1']*100:>9.2f}%")
print("=" * 60)

# Simpan hasil
results = {
    "model"       : MODEL_NAME,
    "threshold"   : THRESHOLD,
    "match_margin": MATCH_MARGIN,
    "total_test"  : len(y_true),
    "metrics"     : {
        k: v.tolist() if isinstance(v, np.ndarray) else v
        for k, v in metrics.items()
    },
    "detail"      : detail_rows,
}
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n📄 Hasil disimpan: {RESULTS_FILE}")

## 7. Visualisasi — Confusion Matrix

In [ ]:
cm     = metrics["confusion_matrix"]
cm_lbl = metrics["cm_labels"]

plt.figure(figsize=(max(6, len(cm_lbl)), max(4, len(labels))))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=cm_lbl,
    yticklabels=labels,
    linewidths=0.5,
)
plt.title(f"Confusion Matrix — {MODEL_NAME}\nThreshold={THRESHOLD}, Margin={MATCH_MARGIN}",
          fontsize=13, pad=15)
plt.xlabel("Predicted", fontsize=11)
plt.ylabel("Actual", fontsize=11)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ml_model/confusion_matrix.png", dpi=150)
plt.show()
print("✅ Confusion matrix disimpan.")

## 8. Visualisasi — Score Distribution per Orang

In [ ]:
from collections import defaultdict

scores_by_person = defaultdict(list)
for row in detail_rows:
    scores_by_person[row["true"]].append(row["best_score"])

fig, ax = plt.subplots(figsize=(8, 4))
for person, scores in scores_by_person.items():
    ax.scatter([person] * len(scores), scores, s=80, label=person, zorder=3)

ax.axhline(THRESHOLD, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold ({THRESHOLD})")
ax.set_title(f"Cosine Similarity Score per Orang — {MODEL_NAME}", fontsize=13)
ax.set_xlabel("Orang (True Label)", fontsize=11)
ax.set_ylabel("Cosine Similarity Score", fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ml_model/score_distribution.png", dpi=150)
plt.show()
print("✅ Score distribution disimpan.")